# 6. Delivery Analysis

## Objective

Evaluate delivery performance by analyzing:

- On-Time Deliveries
- Late Deliveries
- In-Full Deliveries
- OTIF Performance
- Average Delivery Delay
- Monthly Delivery Trends

In [17]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("supply_chain.db")

print("✅ Connected to SQLite Database Successfully!")

✅ Connected to SQLite Database Successfully!


In [18]:
## Overall Delivery Performance
pd.read_sql("""
SELECT

ROUND(AVG("On Time")*100,2) AS on_time_percentage,

ROUND(AVG("In Full")*100,2) AS in_full_percentage,

ROUND(AVG("On Time In Full")*100,2) AS otif_percentage

FROM fact_order_line;
""", conn)

,on_time_percentage,in_full_percentage,otif_percentage
0,71.21,65.93,47.84


In [19]:
## Total Late Deliveries
pd.read_sql("""
SELECT

COUNT(*) AS late_deliveries

FROM fact_order_line

WHERE "On Time"=0;
""", conn)


,late_deliveries
0,6966


In [20]:
## Total On-Time Deliveries
pd.read_sql("""
SELECT

COUNT(*) AS on_time_deliveries

FROM fact_order_line

WHERE "On Time"=1;
""", conn)


,on_time_deliveries
0,17229


In [21]:
##  Total In-Full Deliveries
pd.read_sql("""
SELECT

COUNT(*) AS in_full_deliveries

FROM fact_order_line

WHERE "In Full"=1;
""", conn)


,in_full_deliveries
0,15952


In [22]:
pd.read_sql("""
SELECT

COUNT(*) AS otif_deliveries

FROM fact_order_line

WHERE "On Time" = 1
AND "In Full" = 1;

""", conn)

,otif_deliveries
0,11574


In [23]:
## Average Delivery Delay
pd.read_sql("""
SELECT

ROUND(
AVG(
julianday(actual_delivery_date)
-
julianday(agreed_delivery_date)
),2
) AS average_delay_days

FROM fact_order_line;
""", conn)


,average_delay_days
0,0.42


In [24]:
pd.read_sql("""
SELECT

ROUND(
    COUNT(*) * 100.0 /
    (SELECT COUNT(*) FROM fact_order_line),
    2
) AS otif_percentage

FROM fact_order_line

WHERE "On Time" = 1
AND "In Full" = 1;

""", conn)

,otif_percentage
0,47.84


In [25]:
## Maximum Delivery Delay
pd.read_sql("""
SELECT

MAX(
julianday(actual_delivery_date)
-
julianday(agreed_delivery_date)
) AS maximum_delay_days

FROM fact_order_line;
""", conn)


,maximum_delay_days
0,3.0


In [26]:
## Earliest Delivery
pd.read_sql("""
SELECT

MIN(
julianday(actual_delivery_date)
-
julianday(agreed_delivery_date)
) AS earliest_delivery_days

FROM fact_order_line;
""", conn)


,earliest_delivery_days
0,-1.0


In [27]:
## Delivery Status
pd.read_sql("""
SELECT

CASE

WHEN "On Time"=1
THEN 'On Time'

ELSE 'Late'

END AS delivery_status,

COUNT(*) AS total_orders

FROM fact_order_line

GROUP BY delivery_status;
""", conn)


,delivery_status,total_orders
0,Late,6966
1,On Time,17229


In [28]:
import pandas as pd

order_line = pd.read_csv("../data/fact_order_line.csv")

In [29]:
## Convert the date in Python
import pandas as pd

order_line["order_placement_date"] = pd.to_datetime(
    order_line["order_placement_date"],
    format="%d-%m-%Y"
)

order_line["agreed_delivery_date"] = pd.to_datetime(
    order_line["agreed_delivery_date"],
    format="%d-%m-%Y"
)

order_line["actual_delivery_date"] = pd.to_datetime(
    order_line["actual_delivery_date"],
    format="%d-%m-%Y"
)

In [30]:
order_line["order_placement_date"] = order_line["order_placement_date"].dt.strftime("%Y-%m-%d")
order_line["agreed_delivery_date"] = order_line["agreed_delivery_date"].dt.strftime("%Y-%m-%d")
order_line["actual_delivery_date"] = order_line["actual_delivery_date"].dt.strftime("%Y-%m-%d")

In [31]:
order_line.to_sql(
    "fact_order_line",
    conn,
    if_exists="replace",
    index=False
)

24195

In [32]:
## Monthly OTIF Performance
pd.read_sql("""
SELECT

strftime('%Y-%m', order_placement_date) AS month,

ROUND(
AVG("On Time In Full") * 100,
2
) AS otif_percentage

FROM fact_order_line

GROUP BY month

ORDER BY month;
""", conn)


,month,otif_percentage
0,2025-03,48.00
1,2025-04,47.77
2,2025-05,47.66


In [33]:
## Monthly Order Volume
pd.read_sql("""
SELECT

strftime('%Y-%m',order_placement_date) AS month,

COUNT(DISTINCT order_id) AS total_orders

FROM fact_order_line

GROUP BY month

ORDER BY month;
""", conn)


,month,total_orders
0,2025-03,5407
1,2025-04,5253
2,2025-05,2807


In [34]:
## Monthly Revenue
pd.read_sql("""
SELECT

strftime('%Y-%m',f.order_placement_date) AS month,

ROUND(
SUM(f.delivery_qty*p.price_INR),2
) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id=p.product_id

GROUP BY month

ORDER BY month;
""", conn)

,month,revenue
0,2025-03,236934554.0
1,2025-04,224451516.0
2,2025-05,122578566.0


In [35]:
## Best Performing Month
pd.read_sql("""
SELECT

strftime('%Y-%m', f.order_placement_date) AS month,

ROUND(
SUM(f.delivery_qty * p.price_INR),
2
) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id = p.product_id

GROUP BY month

ORDER BY revenue DESC

LIMIT 1;
""", conn)


,month,revenue
0,2025-03,236934554.0


In [36]:
## Worst Performing Month
pd.read_sql("""
SELECT

strftime('%Y-%m', f.order_placement_date) AS month,

ROUND(
SUM(f.delivery_qty * p.price_INR),
2
) AS revenue

FROM fact_order_line f

JOIN dim_products p

ON f.product_id = p.product_id

GROUP BY month

ORDER BY revenue ASC

LIMIT 1;
""", conn)


,month,revenue
0,2025-05,122578566.0


In [37]:
## City-wise Delivery Performance
pd.read_sql("""
SELECT

c.city,

ROUND(
AVG("On Time")*100,
2
) AS on_time_percentage,

ROUND(
AVG("In Full")*100,
2
) AS in_full_percentage,

ROUND(
AVG(
CASE
WHEN "On Time"=1
AND "In Full"=1
THEN 1
ELSE 0
END
)*100,
2
) AS otif_percentage

FROM fact_order_line f

JOIN dim_customers c
ON f.customer_id=c.customer_id

GROUP BY c.city

ORDER BY otif_percentage DESC;

""", conn)

,city,on_time_percentage,in_full_percentage,otif_percentage
0,"New Jersey, US",73.49,66.32,49.82
1,Ahmedabad,70.40,67.44,48.44
2,Vadodara,69.91,64.02,45.36


In [38]:
## Category-wise Delivery Performance
pd.read_sql("""
SELECT

p.category,

ROUND(
AVG("On Time")*100,
2
) AS on_time_percentage,

ROUND(
AVG("In Full")*100,
2
) AS in_full_percentage,

ROUND(
AVG(
CASE
WHEN "On Time"=1
AND "In Full"=1
THEN 1
ELSE 0
END
)*100,
2
) AS otif_percentage

FROM fact_order_line f

JOIN dim_products p
ON f.product_id=p.product_id

GROUP BY p.category

ORDER BY otif_percentage DESC;

""", conn)


,category,on_time_percentage,in_full_percentage,otif_percentage
0,Food,71.99,66.18,48.57
1,beverages,71.82,66.22,48.06
2,Dairy,70.86,65.80,47.59


## Business Insights

- **March 2025** recorded the highest order volume with **5,407 orders**, indicating the strongest customer demand during the analysis period.

- Order volume decreased slightly to **5,253 orders** in **April 2025**, suggesting relatively stable business performance.

- **May 2025** experienced a significant drop to **2,807 orders**, representing the lowest monthly order volume.

- The decline in May may indicate seasonal demand changes, reduced customer activity, or business operational factors that require further investigation.

- Monitoring monthly order trends helps improve demand forecasting, inventory planning, and production scheduling.

### Recommendations

- Investigate the reasons behind the decline in customer orders during May.
- Continue monitoring OTIF to improve customer service levels.
- Focus on increasing demand through marketing initiatives while maintaining operational efficiency.
- Improve logistics and inventory planning to increase the OTIF percentage beyond 50%.


